# Tahap 4b: Evaluasi Komparatif Model CBF Alternatif (Saran 1)
Notebook ini melatih dan mengevaluasi 4 variasi model Content-Based Filtering (CBF) dengan menggabungkan konteks **Nama resep** dan **Tag masakan** ke dalam representasi model:
1. **TF-IDF (Baseline)**: Teks gabungan (Nama + Bahan + Tag) dengan pembobotan TF-IDF.
2. **Word2Vec Embeddings**: Dense representation dilatih pada seluruh token kata di `combined_text` (Nama + Bahan + Tag).
3. **Multi-Hot Jaccard Similarity**: Kesamaan overlap himpunan berbasis gabungan list kategori **Bahan + Tag**.
4. **Node2Vec Co-occurrence Embeddings**: Graf relasi co-occurrence antar entitas kategori **Bahan + Tag**.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# Daftarkan direktori agar modul lokal bisa dibaca
sys.path.append('.')
sys.path.append(str(Path.cwd().parent))

from feature_extractor import load_recipes, extract_text_features, parse_ast_list
from models import TFIDFRecommender, Word2VecRecommender, JaccardRecommender, Node2VecRecommender

MODEL_DIR = Path("outputs/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("outputs/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
print("1. Memuat dan Mempersiapkan Dataset...")
df_recipes = load_recipes()
df_recipes = extract_text_features(df_recipes)

# Deteksi kolom parsed ingredients secara dinamis
ing_parsed_col = 'ingredient_names_parsed' if 'ingredient_names_parsed' in df_recipes.columns else 'ingredients_parsed'

# Tokenisasi teks lengkap (Nama + Bahan + Tag) untuk Word2Vec
df_recipes['w2v_tokens'] = df_recipes['combined_text'].fillna('').apply(lambda x: str(x).split())

# Penggabungan kategori Bahan + Tag untuk Jaccard & Node2Vec co-occurrence
df_recipes['cbf_tokens'] = df_recipes[ing_parsed_col] + df_recipes['tags_parsed']

print(f"Total data resep siap latih: {len(df_recipes)}")
df_recipes[['id', 'cf_idx', 'combined_text', 'w2v_tokens', 'cbf_tokens']].head(2)

1. Memuat dan Mempersiapkan Dataset...
Total recipes raw: 231637
Filtered recipes matching CF split: 75017
Total data resep siap latih: 75017


,id,cf_idx,combined_text,w2v_tokens,cbf_tokens
0,40,0,best lemonade sugar lemons rind of fresh water...,"[best, lemonade, sugar, lemons, rind, of, fres...","[sugar, lemons, rind of, fresh water, fresh le..."
1,49,1,chicken breasts lombardi fresh mushroom butter...,"[chicken, breasts, lombardi, fresh, mushroom, ...","[fresh mushroom, butter, marsala, chicken brot..."


In [3]:
print("2. Melatih Model 1: TF-IDF (Baseline)...")
tfidf_model = TFIDFRecommender(max_features=5000)
tfidf_model.fit(df_recipes, text_col='combined_text', id_col='id')
tfidf_model.save(MODEL_DIR / "tfidf_cbf_model.pkl")
print("Model TF-IDF selesai dilatih.")

2. Melatih Model 1: TF-IDF (Baseline)...
Fitting TF-IDF model on 75017 recipes...
TF-IDF Matrix shape: (75017, 5000)
Model saved to outputs\models\tfidf_cbf_model.pkl
Model TF-IDF selesai dilatih.


In [4]:
print("3. Melatih Model 2: Word2Vec Embeddings (Nama + Bahan + Tag)...")
w2v_model = Word2VecRecommender(vector_size=100, window=5, min_count=1, sg=1, seed=42)
w2v_model.fit(df_recipes, ingredients_col='w2v_tokens', id_col='id')
w2v_model.save(MODEL_DIR / "w2v_cbf_model.pkl")
print("Model Word2Vec selesai dilatih.")

3. Melatih Model 2: Word2Vec Embeddings (Nama + Bahan + Tag)...
Training Word2Vec model on 75017 recipes...
Recipe vectors shape: (75017, 100)
Model saved to outputs\models\w2v_cbf_model.pkl
Model Word2Vec selesai dilatih.


In [5]:
print("4. Mempersiapkan Model 3: Multi-Hot Jaccard Similarity (Bahan + Tag)...")
jaccard_model = JaccardRecommender()
jaccard_model.fit(df_recipes, ingredients_col='cbf_tokens', id_col='id')
jaccard_model.save(MODEL_DIR / "jaccard_cbf_model.pkl")
print("Model Jaccard selesai dipersiapkan.")

4. Mempersiapkan Model 3: Multi-Hot Jaccard Similarity (Bahan + Tag)...
Fitting Jaccard model on 75017 recipes...
Jaccard index built for 75017 recipes.
Model saved to outputs\models\jaccard_cbf_model.pkl
Model Jaccard selesai dipersiapkan.


In [6]:
print("5. Melatih Model 4: Node2Vec Co-occurrence Embeddings (Bahan + Tag)...")
node2vec_model = Node2VecRecommender(vector_size=100, num_walks=5, walk_length=15, window=5, sg=1, seed=42)
node2vec_model.fit(df_recipes, ingredients_col='cbf_tokens', id_col='id')
node2vec_model.save(MODEL_DIR / "node2vec_cbf_model.pkl")
print("Model Node2Vec selesai dilatih.")

5. Melatih Model 4: Node2Vec Co-occurrence Embeddings (Bahan + Tag)...
Building co-occurrence graph for 75017 recipes...
Graph built with 6423 nodes and 902449 edges.
Precomputing transition tables...
Generating random walks (num_walks=5, walk_length=15)...
Generated 32115 random walks. Training Word2Vec embeddings...
Recipe vectors shape: (75017, 100)
Model saved to outputs\models\node2vec_cbf_model.pkl
Model Node2Vec selesai dilatih.


In [7]:
from cf.data_prep import load_cf_splits, build_user_history, prepare_loo_eval, get_matrix_dims
from cf.evaluator import evaluate_loo, metrics_to_dataframe

print("6. Menyiapkan Dataset Evaluasi Leave-One-Out (LOO)...")
cf_train, cf_val, cf_test = load_cf_splits()
user_history = build_user_history(cf_train, cf_val)
n_users, n_items = get_matrix_dims(cf_train, cf_val, cf_test)

# Menyiapkan data LOO dengan 99 item negatif
loo_data = prepare_loo_eval(cf_test, user_history, n_items, n_negatives=99, seed=42)
cbf_map = dict(zip(df_recipes['cf_idx'], df_recipes['id']))
print("Data evaluasi LOO siap.")

6. Menyiapkan Dataset Evaluasi Leave-One-Out (LOO)...
[data_prep] CF LOO split loaded -- Train: 622,730  Val: 19,123  Test: 19,123
[data_prep] Matrix dims -- Users: 19,123  Items: 75,017
[data_prep] LOO users for evaluation: 19,123
[data_prep] LOO eval records ready: 19,123
Data evaluasi LOO siap.


In [8]:
# 7. Wrapper Classes untuk Evaluasi Komparatif

class TFIDFUserRecommenderWrapper:
    def __init__(self, model, user_hist, id_map):
        self.model = model
        self.user_history = user_hist
        self.id_map = id_map
        self.mat = self.model.tfidf_matrix
        self.valid_history = {}
        for u, items in self.user_history.items():
            valid_idx = [self.model.item_id_to_idx[self.id_map[i]] 
                         for i in items if self.id_map.get(i) in self.model.item_id_to_idx]
            if valid_idx:
                self.valid_history[u] = valid_idx
                
    def score_candidates(self, u: int, candidates: np.ndarray) -> np.ndarray:
        hist_idx = self.valid_history.get(u)
        if not hist_idx:
            return np.zeros(len(candidates))
        cand_idx = []
        valid_mask = []
        for c in candidates:
            real_id = self.id_map.get(c)
            if real_id is not None and real_id in self.model.item_id_to_idx:
                cand_idx.append(self.model.item_id_to_idx[real_id])
                valid_mask.append(True)
            else:
                cand_idx.append(0)
                valid_mask.append(False)
        hist_vecs = self.mat[hist_idx]
        cand_vecs = self.mat[cand_idx]
        sims = cosine_similarity(hist_vecs, cand_vecs)
        scores = sims.max(axis=0) * np.array(valid_mask)
        return scores


class Word2VecUserRecommenderWrapper:
    def __init__(self, model, user_hist, id_map):
        self.model = model
        self.user_history = user_hist
        self.id_map = id_map
        self.mat = self.model.recipe_vectors
        self.valid_history = {}
        for u, items in self.user_history.items():
            valid_idx = [self.model.item_id_to_idx[self.id_map[i]] 
                         for i in items if self.id_map.get(i) in self.model.item_id_to_idx]
            if valid_idx:
                self.valid_history[u] = valid_idx
                
    def score_candidates(self, u: int, candidates: np.ndarray) -> np.ndarray:
        hist_idx = self.valid_history.get(u)
        if not hist_idx:
            return np.zeros(len(candidates))
        cand_idx = []
        valid_mask = []
        for c in candidates:
            real_id = self.id_map.get(c)
            if real_id is not None and real_id in self.model.item_id_to_idx:
                cand_idx.append(self.model.item_id_to_idx[real_id])
                valid_mask.append(True)
            else:
                cand_idx.append(0)
                valid_mask.append(False)
        hist_vecs = self.mat[hist_idx]
        cand_vecs = self.mat[cand_idx]
        sims = cosine_similarity(hist_vecs, cand_vecs)
        scores = sims.max(axis=0) * np.array(valid_mask)
        return scores


class JaccardUserRecommenderWrapper:
    def __init__(self, model, user_hist, id_map):
        self.model = model
        self.user_history = user_hist
        self.id_map = id_map
        self.valid_history_ids = {}
        for u, items in self.user_history.items():
            valid_ids = [self.id_map[i] for i in items if self.id_map.get(i) in self.model.recipe_ingredients]
            if valid_ids:
                self.valid_history_ids[u] = valid_ids
                
    def score_candidates(self, u: int, candidates: np.ndarray) -> np.ndarray:
        hist_ids = self.valid_history_ids.get(u)
        if not hist_ids:
            return np.zeros(len(candidates))
        scores = []
        for c in candidates:
            real_id = self.id_map.get(c)
            if real_id is None or real_id not in self.model.recipe_ingredients:
                scores.append(0.0)
                continue
            cand_set = self.model.recipe_ingredients[real_id]
            max_sim = 0.0
            for h_id in hist_ids:
                hist_set = self.model.recipe_ingredients[h_id]
                intersection = len(cand_set.intersection(hist_set))
                union = len(cand_set.union(hist_set))
                sim = intersection / union if union > 0 else 0.0
                if sim > max_sim:
                    max_sim = sim
            scores.append(max_sim)
        return np.array(scores)

In [9]:
print("8. Memulai Evaluasi untuk Semua Model...")

results = {}

# 8.1 Evaluasi TF-IDF
print("Evaluating TF-IDF...")
eval_tfidf = TFIDFUserRecommenderWrapper(tfidf_model, user_history, cbf_map)
results["CBF_TFIDF_Baseline"] = evaluate_loo(eval_tfidf, loo_data, k_values=(5, 10, 20), verbose=False)

# 8.2 Evaluasi Word2Vec
print("Evaluating Word2Vec...")
eval_w2v = Word2VecUserRecommenderWrapper(w2v_model, user_history, cbf_map)
results["CBF_Word2Vec"] = evaluate_loo(eval_w2v, loo_data, k_values=(5, 10, 20), verbose=False)

# 8.3 Evaluasi Jaccard
print("Evaluating Jaccard...")
eval_jaccard = JaccardUserRecommenderWrapper(jaccard_model, user_history, cbf_map)
results["CBF_Jaccard"] = evaluate_loo(eval_jaccard, loo_data, k_values=(5, 10, 20), verbose=False)

# 8.4 Evaluasi Node2Vec
print("Evaluating Node2Vec...")
eval_node2vec = Word2VecUserRecommenderWrapper(node2vec_model, user_history, cbf_map)
results["CBF_Node2Vec"] = evaluate_loo(eval_node2vec, loo_data, k_values=(5, 10, 20), verbose=False)

print("Evaluasi selesai.")

8. Memulai Evaluasi untuk Semua Model...
Evaluating TF-IDF...
Evaluating Word2Vec...
Evaluating Jaccard...
Evaluating Node2Vec...
Evaluasi selesai.


In [10]:
print("9. Kesimpulan Hasil Evaluasi Komparatif Model CBF:")
results_df = metrics_to_dataframe(results)
display(results_df.style.highlight_max(color='lightgreen', axis=0))

# Simpan hasil perbandingan ke CSV
results_file = RESULTS_DIR / "cbf_alternatives_results.csv"
results_df.to_csv(results_file)
print(f"\nHasil komparatif berhasil disimpan di: {results_file}")

9. Kesimpulan Hasil Evaluasi Komparatif Model CBF:


,HR@5,HR@10,HR@20,MRR
Model,,,,
CBF_TFIDF_Baseline,0.143074,0.231240,0.374157,0.113672
CBF_Word2Vec,0.113894,0.195785,0.330806,0.092862
CBF_Jaccard,0.132354,0.221304,0.359776,0.105802
CBF_Node2Vec,0.115202,0.202531,0.345291,0.095363



Hasil komparatif berhasil disimpan di: outputs\results\cbf_alternatives_results.csv
